In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import re
from collections import Counter
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import matthews_corrcoef
from collections import defaultdict
import logomaker as lm
# ===========================
# 1) PROTSCALE-LIKE DICTIONARIES
# ===========================

# Kyte–Doolittle hydrophobicity scale
KD = {
    'A': 1.8,  'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8,  'K': -3.9, 'M': 1.9,  'F': 2.8,  'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

# Zimmerman-like polarity scale (approximation)
POLARITY = {
    'A': 0.00, 'C': 1.48, 'D': 49.7, 'E': 49.9, 'F': 0.35,
    'G': 0.00, 'H': 51.6, 'I': 0.13, 'K': 49.5, 'L': 0.13,
    'M': 1.43, 'N': 3.38, 'P': 1.58, 'Q': 3.53, 'R': 52.0,
    'S': 1.67, 'T': 1.66, 'V': 0.13, 'W': 2.10, 'Y': 1.61
}

# Relative side-chain volumes
VOLUME = {
    'A': 31,  'C': 55,  'D': 54,  'E': 83,  'F': 132,
    'G': 3,   'H': 96,  'I': 111, 'K': 119, 'L': 111,
    'M': 105, 'N': 56,  'P': 32.5,'Q': 85,  'R': 124.5,
    'S': 32,  'T': 61,  'V': 84,  'W': 170, 'Y': 136
}

# Approximate side-chain charge at pH ~7
CHARGE_PH7 = {
    'A': 0.0, 'C': 0.0, 'D': -1.0, 'E': -1.0, 'F': 0.0,
    'G': 0.0, 'H': +0.1,'I': 0.0,  'K': +1.0, 'L': 0.0,
    'M': 0.0, 'N': 0.0, 'P': 0.0,  'Q': 0.0,  'R': +1.0,
    'S': 0.0, 'T': 0.0, 'V': 0.0,  'W': 0.0,  'Y': 0.0
}

# List of standard amino acids in a consistent order
AMINOACIDS = ['A','C','D','E','F','G','H','I','K','L',
              'M','N','P','Q','R','S','T','V','W','Y']


# ===========================
# 2) HELPER FUNCTIONS FOR SCALES
# ===========================

def _map_scale(seq, scale):
    """
    Map each residue in the sequence to a numeric value
    according to a physicochemical scale dictionary.
    Residues not in the scale are skipped.
    """
    return [scale[a] for a in seq if a in scale]


def _sliding_means(vals, w):
    """
    Compute mean values over all sliding windows of length w
    across a list of numeric values (e.g. KD hydrophobicity).
    Returns a list of window means (length n-w+1), or [] if
    the sequence is shorter than the window.
    """
    n = len(vals)
    if w <= 0 or n < w:
        return []
    # Prefix-sum trick for O(n) sliding-window mean
    c = np.cumsum([0.0] + vals)
    return [(c[i+w] - c[i]) / w for i in range(n - w + 1)]


def _longest_run_by_threshold(vals, thr):
    """
    Compute the longest consecutive run of values >= thr.
    Used to quantify hydrophobic stretches (KD >= threshold).
    """
    best = cur = 0
    for v in vals:
        if v >= thr:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best


def extract_scaled_features_first70(seq, win=7, nterm_len=70, kd_thr=1.6):
    """
    Extract physicochemical (ProtScale-like) features from the
    first nterm_len residues of a sequence (default 70 aa).

    For each scale (KD, POLARITY, VOLUME, CHARGE) it computes:
      - mean over first 70 residues
      - max mean over sliding windows of length `win`

    Additional features:
      - longest_hydrophobic_run_70 (KD >= kd_thr)
      - entropy_70 (Shannon entropy of composition)
      - num_disulfide_motifs_70 ("CxxC" with 2–5 variable positions)
      - amino acid frequencies in the first 70 residues
      - total sequence length (length of the full protein)
    """
    seq = (seq or "").strip().upper()
    L = len(seq)
    nseq = seq[:min(L, nterm_len)]
    nlen = len(nseq)

    # Handle empty sequences safely
    if nlen == 0:
        feats = {
            "length": L,
            "kd_mean_70": 0.0,           "kd_win7_max": 0.0,
            "polarity_mean_70": 0.0,     "polarity_win7_max": 0.0,
            "volume_mean_70": 0.0,       "volume_win7_max": 0.0,
            "charge_mean_70": 0.0,       "charge_win7_max": 0.0,
            "longest_hydrophobic_run_70": 0,
            "entropy_70": 0.0,
            "num_disulfide_motifs_70": 0,
        }
        # Zero composition for all amino acids
        feats.update({f"%{aa}": 0.0 for aa in AMINOACIDS})
        return feats

    # Map residue sequence to numeric vectors for each scale
    kd_vals   = _map_scale(nseq, KD)
    pol_vals  = _map_scale(nseq, POLARITY)
    vol_vals  = _map_scale(nseq, VOLUME)
    chg_vals  = _map_scale(nseq, CHARGE_PH7)

    # Global means on the N-terminal window
    kd_mean   = float(np.mean(kd_vals))   if kd_vals  else 0.0
    pol_mean  = float(np.mean(pol_vals))  if pol_vals else 0.0
    vol_mean  = float(np.mean(vol_vals))  if vol_vals else 0.0
    chg_mean  = float(np.mean(chg_vals))  if chg_vals else 0.0

    # Sliding-window mean maxima
    kd_win  = _sliding_means(kd_vals,  win);   kd_wmax  = float(np.max(kd_win))  if kd_win  else 0.0
    pol_win = _sliding_means(pol_vals, win);   pol_wmax = float(np.max(pol_win)) if pol_win else 0.0
    vol_win = _sliding_means(vol_vals, win);   vol_wmax = float(np.max(vol_win)) if vol_win else 0.0
    chg_win = _sliding_means(chg_vals, win);   chg_wmax = float(np.max(chg_win)) if chg_win else 0.0

    # Longest hydrophobic run based on KD threshold
    longest_hydro = _longest_run_by_threshold(kd_vals, kd_thr)

    # Shannon entropy over composition
    counts = Counter(nseq)
    probs = [c / nlen for c in counts.values() if c > 0]
    entropy = -sum(p * math.log2(p) for p in probs)

    # Rough count of disulfide-like motifs "Cxx...xC" in the first 70 aa
    num_disulfide = len(re.findall(r"C.{2,5}C", nseq))

    # Amino acid frequencies over the N-terminal fragment
    aa_freq = {f"%{aa}": counts.get(aa, 0) / nlen for aa in AMINOACIDS}

    # Collect all features in a single dictionary
    feats = {
        "length": L,
        "kd_mean_70": kd_mean,
        f"kd_win{win}_max": kd_wmax,
        "polarity_mean_70": pol_mean,
        f"polarity_win{win}_max": pol_wmax,
        "volume_mean_70": vol_mean,
        f"volume_win{win}_max": vol_wmax,
        "charge_mean_70": chg_mean,
        f"charge_win{win}_max": chg_wmax,
        "longest_hydrophobic_run_70": longest_hydro,
        "entropy_70": entropy,
        "num_disulfide_motifs_70": num_disulfide,
    }
    feats.update(aa_freq)
    return feats


# ===========================
# 3) PSWM FUNCTIONS
# ===========================

def init_matrix(window):
    """
    Initialize a position-specific count matrix (PSPM) for 20 amino acids,
    with a given window length. Each position starts with a pseudocount = 1.
    """
    aminoacids = ["A","R","N","D","C","Q","E","G","H","I",
                  "L","K","M","F","P","S","T","W","Y","V"]
    dict_mat = {}
    for aa in aminoacids:
        dict_mat[aa] = [1 for _ in range(window)]
    return dict_mat


def compute_pswm(matrix, list_seq):
    """
    Convert a pseudocount matrix (PSPM) into a position-specific weight
    matrix (PSWM) using log-odds with background frequencies (SwissProt).

    For each sequence in list_seq:
      - increment the counts along the first `pswm_len` positions
    Then convert counts to log2( (count / div) / bg_freq )
    where `div = len(list_seq) + 20` and bg_freq is amino acid
    background probability.
    """
    diz_swp = {'A':0.08, 'R':0.06, 'N':0.04, 'D':0.06, 'C':0.01,
               'Q':0.04, 'E':0.07, 'G':0.07, 'H':0.02, 'I':0.06,
               'L':0.10, 'K':0.06, 'M':0.02, 'F':0.04, 'P':0.05,
               'S':0.07, 'T':0.05, 'W':0.01, 'Y':0.03, 'V':0.07}
    pswm_len = len(next(iter(matrix.values())))

    # Count occurrences per position
    for seq in list_seq:
        for index, res in enumerate(seq[:pswm_len]):
            if res not in matrix:
                continue
            matrix[res][index] += 1

    # Normalization divisor: sequences + pseudocounts
    div = int(len(list_seq)) + 20

    # Convert to log-odds weights using background frequencies
    for key in matrix:
        matrix[key] = [np.log2(x / (div * diz_swp[key])) for x in matrix[key]]

    return matrix


def highest_pswm_scores_aligned(pswm, seqs, window_size=15):
    """
    For each sequence, slide a window of length `window_size`
    and calculate the PSWM log-odds score for each window.
    Keep the maximum score as the feature for that sequence.
    Returns an array of shape (n_sequences, 1).
    """
    scores = []
    for seq in seqs:
        seq = (seq or "").upper()

        # If the sequence is shorter than the window, assign a score of 0
        if len(seq) < window_size:
            scores.append(0.0)
            continue

        best = float("-inf")
        # Slide across the sequence
        for i in range(len(seq) - window_size + 1):
            w = seq[i:i+window_size]
            s = 0.0
            for pos, aa in enumerate(w):
                if aa in pswm:
                    s += pswm[aa][pos]
            if s > best:
                best = s
        scores.append(best)

    return np.array(scores, dtype=float).reshape(-1, 1)


# ===========================
# 4) PROLINE & SCALE-BASED FEATURES
# ===========================

# List of keys for the 10 physicochemical features (order matters)
_PHYS_KEYS = [
    "kd_mean_70", "kd_win7_max",
    "polarity_mean_70", "polarity_win7_max",
    "volume_mean_70", "volume_win7_max",
    "charge_mean_70", "charge_win7_max",
    "longest_hydrophobic_run_70", "entropy_70"
]

# Keys for the 20 AA frequency features, e.g. "%A", "%C", ...
_AA_KEYS = [f"%{aa}" for aa in AMINOACIDS]


def extract_scale_window_features(seq, nterm_len=70, win=7, kd_thr=1.6):
    """
    Wrapper around extract_scaled_features_first70() that flattens
    its dictionary output into a fixed-length numeric vector of size 30:
      * 10 physicochemical descriptors
      * 20 amino acid frequency features
    """
    d = extract_scaled_features_first70(seq, win=win, nterm_len=nterm_len, kd_thr=kd_thr)
    vec = [d[k] for k in _PHYS_KEYS] + [d[k] for k in _AA_KEYS]
    return np.array(vec, dtype=float)


def proline_matrix_numpy(seqs, nterm_len=70):
    """
    For each sequence, compute:
      - the number of 'P' (Proline) residues in the first nterm_len positions
      - the Proline density (count / N-terminal length)
    Returns an array of shape (n_sequences, 2).
    """
    results = []
    for seq in seqs:
        s = (seq or "").upper()
        first = s[:nterm_len]
        p_count = first.count("P")
        length = len(first)
        p_density = (p_count / length) if length > 0 else 0.0
        results.append([p_count, p_density])
    return np.array(results, dtype=float)


def build_feature_matrix(seqs, labels, pswm, pswm_len=15, nterm_len=70):
    """
    Build the full feature matrix for a set of sequences:

      Features per sequence:
        - 2 Proline features (count, density)
        - 1 PSWM sliding-window max score
        - 30 scale-based features

      Total: 33 features.
      The last column is the binary label (0/1).

    Returns:
      M: (n, 34) -> [features..., label]
      X: (n, 33) -> feature matrix only
      y: (n,)    -> label vector
    """
    # (n, 1) PSWM scores
    pswm_scores = highest_pswm_scores_aligned(pswm, seqs, window_size=pswm_len)
    # (n, 2) Proline features
    pro_mat = proline_matrix_numpy(seqs, nterm_len=nterm_len)
    # (n, 30) scale-based features
    scale_feats = np.vstack([
        extract_scale_window_features(s, nterm_len=nterm_len) for s in seqs
    ])

    # Concatenate features horizontally: [Pro(2) | PSWM(1) | Scale(30)] = 33
    X = np.hstack([pro_mat, pswm_scores, scale_feats])
    # Label column
    y = np.array(labels, dtype=int).reshape(-1, 1)
    # Final matrix with label as last column
    M = np.hstack([X, y])
    return M, X, y.ravel()


# ===========================
# 5) LOAD DATA FILES + TRAIN/VAL/TEST SPLIT
# ===========================

files = [
    "/home/anomalocaris/LB2_project_group_9/SVM/subset1_arricchito.tsv",
    "/home/anomalocaris/LB2_project_group_9/SVM/subset2_arricchito.tsv",
    "/home/anomalocaris/LB2_project_group_9/SVM/subset3_arricchito.tsv",
    "/home/anomalocaris/LB2_project_group_9/SVM/subset4_arricchito.tsv",
    "/home/anomalocaris/LB2_project_group_9/SVM/subset5_arricchito.tsv",
    "/home/anomalocaris/LB2_project_group_9/Data Preparation/benchmark_arricchito.tsv"
]

# all_seqs[i]  -> list of sequences for file i
# all_labels[i] -> list of binary labels (0/1) for file i
all_seqs = []
all_labels = []

for i, f in enumerate(files, start=1):
    print(f"[{i}/6] Loading {f}")
    dataset = pd.read_csv(f, sep="\t")

    # Basic sanity checks for required columns
    if 'SP cleavage' not in dataset.columns:
        raise ValueError(f"Column 'SP cleavage' is missing in {f}")
    if 'Sequence' not in dataset.columns:
        raise ValueError(f"Column 'Sequence' is missing in {f}")

    # SP cleavage column: "False" -> 0, everything else -> 1
    dataset['positive/negative'] = np.where(dataset['SP cleavage'] == 'False', 0, 1).astype(int)

    seqs = dataset["Sequence"].fillna("").astype(str).tolist()
    labels = dataset["positive/negative"].astype(int).tolist()

    all_seqs.append(seqs)
    all_labels.append(labels)

print(f"\nLoaded {len(all_seqs)} datasets")
for i, seq_list in enumerate(all_seqs):
    print(f"File {i+1}: {len(seq_list)} sequences")

# Fixed split:
#   Train = subsets 1–4
#   Val   = subset 5
#   Test  = benchmark file (6th)
train_seqs   = all_seqs[1] + all_seqs[2] + all_seqs[3] + all_seqs[4]
train_labels = all_labels[1] + all_labels[2] + all_labels[3] + all_labels[4]

val_seqs   = all_seqs[0]
val_labels = all_labels[0]

test_seqs   = all_seqs[5]
test_labels = all_labels[5]

print("\nSplit sizes:")
print(f"  Train: {len(train_seqs)}")
print(f"  Val:   {len(val_seqs)}")
print(f"  Test:  {len(test_seqs)}")

# ===========================
# 6) TRAIN PSWM USING ONLY POSITIVE TRAINING EXAMPLES
# ===========================

pswm_len = 15
# Extract only sequences with label = 1 from the training set
pos_train_seqs = [s for s, y in zip(train_seqs, train_labels) if y == 1]
print(f"\nUsing {len(pos_train_seqs)} positive training sequences to train the PSWM")

if len(pos_train_seqs) == 0:
    # Safety fallback: if no positives exist, use all sequences
    print("  No positives in training set — using ALL training sequences for PSWM.")
    pos_train_seqs = train_seqs

# Initialize matrix with pseudocounts and compute PSWM
pswm = init_matrix(pswm_len)
pswm = compute_pswm(pswm, pos_train_seqs)

# ===========================
# 7) BUILD FEATURE MATRICES FOR TRAIN / VAL / TEST
# ===========================

M_train, X_train, y_train = build_feature_matrix(train_seqs, train_labels, pswm, pswm_len=pswm_len)
M_val,   X_val,   y_val   = build_feature_matrix(val_seqs,   val_labels,   pswm, pswm_len=pswm_len)
M_test,  X_test,  y_test  = build_feature_matrix(test_seqs,  test_labels,  pswm, pswm_len=pswm_len)

FEATURES = X_train.shape[1]
print(f"\nDetected {FEATURES} total features (indices 0..{FEATURES-1}).")

# ===========================
# 8) USE ONLY THE PRE-SELECTED FEATURES
# ===========================

# Indices of features selected a priori (from a previous feature selection step)
best_features = np.array([
    2, 3, 4, 5, 6, 7, 8, 9,
    10, 11, 12, 13, 14, 15, 16, 17, 18,
    20, 21, 22, 25, 27, 28, 29, 30, 31
])

# Sanity check: all indices must exist
if best_features.max() >= FEATURES:
    raise ValueError(f"Some selected feature indices are >= number of features ({FEATURES}).")

# Slice feature matrices to keep only the selected columns
X_train_sel = X_train[:, best_features]
X_val_sel   = X_val[:,   best_features]
X_test_sel  = X_test[:,  best_features]

print(f"Using {len(best_features)} selected features.")

# ===========================
# 9) GRID SEARCH FOR SVM ON TRAIN → VALIDATION
# ===========================

# Parameter grids for C and gamma of the RBF SVM
C_grid = [0.1, 1.0, 10.0, 100.0]
gamma_grid = ["scale", 0.01, 0.1, 1.0]

# Pipeline: standardize features, then SVM with RBF kernel
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

best_mcc = -1
best_params = None

# Manual grid search over all combinations of C and gamma
for params in ParameterGrid({"C": C_grid, "gamma": gamma_grid}):
    pipeline.set_params(svm__C=params["C"], svm__gamma=params["gamma"])
    pipeline.fit(X_train_sel, y_train)
    y_pred_val = pipeline.predict(X_val_sel)
    mcc_val = matthews_corrcoef(y_val, y_pred_val)

    if mcc_val > best_mcc:
        best_mcc = mcc_val
        best_params = params

print(f"\nBest SVM params on selected features: {best_params}, Val MCC = {best_mcc:.3f}")

# ===========================
# 10) TRAIN ON TRAIN+VAL AND EVALUATE ON TEST (BENCHMARK)
# ===========================

# Concatenate training and validation sets with selected features
X_combined_sel = np.vstack([X_train_sel, X_val_sel])
y_combined = np.concatenate([y_train, y_val])

# Refit the pipeline with the best hyperparameters on train+val
pipeline.set_params(svm__C=best_params["C"], svm__gamma=best_params["gamma"])
pipeline.fit(X_combined_sel, y_combined)

# Evaluate on benchmark (test) set
y_pred_test = pipeline.predict(X_test_sel)
test_mcc = matthews_corrcoef(y_test, y_pred_test)
print(f"\nFINAL TEST MCC (benchmark, selected features) = {test_mcc:.3f}")


[1/6] Loading subset1_arricchito.tsv
[2/6] Loading subset2_arricchito.tsv
[3/6] Loading subset3_arricchito.tsv
[4/6] Loading subset4_arricchito.tsv
[5/6] Loading subset5_arricchito.tsv
[6/6] Loading /home/anomalocaris/LB2_project_group_9/Data Preparation/benchmark_arricchito.tsv

Loaded 6 datasets
File 1: 1605 sequences
File 2: 1605 sequences
File 3: 1604 sequences
File 4: 1603 sequences
File 5: 1603 sequences
File 6: 2006 sequences

Split sizes:
  Train: 6415
  Val:   1605
  Test:  2006

Using 698 positive training sequences to train the PSWM

Detected 33 total features (indices 0..32).
Using 26 selected features.

Best SVM params on selected features: {'C': 10.0, 'gamma': 0.01}, Val MCC = 0.774

FINAL TEST MCC (benchmark, selected features) = 0.752


The decrease in performance from an MCC above 0.80 in the original five-fold cross-validation to approximately 0.77 in the current fixed train/validation/test setup is both expected and methodologically justified. In the former approach, each subset of the data was used as training material in multiple folds, which allowed the model to be indirectly exposed to the entire dataset. This inevitably produces optimistic estimates, because the validation folds are highly similar in distribution to the folds used for training. In contrast, the present evaluation separates the data much more strictly: subsets 1–4 are used exclusively for training, subset 5 is used only for validation and hyperparameter tuning, and the final benchmark dataset—coming from a different source and exhibiting different sequence distributions—is reserved for true out-of-distribution testing. Under these conditions, a drop in MCC is a natural consequence of removing information leakage and increasing the difficulty of the task. Importantly, an MCC around 0.77 on an external benchmark is a strong and realistic result for a classical machine-learning model based on engineered features (PSWM, physicochemical scales, and proline statistics), especially given the heterogeneity and limited size of the training material. The new score therefore represents a more reliable estimate of the model’s real-world generalization ability rather than a deterioration of its intrinsic quality.
